# 008: Batch Normalization — Internal covariate shift, per-layer normalization math, and learnable affine parameters

## 1. INTERNAL COVARIATE SHIFT
- **Problem (Ioffe & Szegedy, 2015)**: As training updates weights in lower layers, the input distribution to each subsequent layer shifts constantly, forcing higher layers to continuously re-adapt.
- **Solution**: Normalize each layer's pre-activations to zero mean and unit variance **within each mini-batch**.

## 2. BATCH NORMALIZATION EQUATIONS
- For a mini-batch $\{z_1, \ldots, z_m\}$ at a given layer:
  $$\mu_B = \frac{1}{m}\sum_{i=1}^{m} z_i$$
  $$\sigma_B^2 = \frac{1}{m}\sum_{i=1}^{m}(z_i - \mu_B)^2$$
  $$\hat{z}_i = \frac{z_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$$
  $$y_i = \gamma \hat{z}_i + \beta$$
- $\gamma$ (scale) and $\beta$ (shift) are **learnable parameters** that allow the network to undo the normalization if needed.
- $\epsilon$ is a small constant (e.g., $10^{-5}$) for numerical stability.

---


In [ ]:
import numpy as np

class BatchNorm1D:
    """From-scratch batch normalization for fully connected layers."""
    def __init__(self, num_features, epsilon=1e-5, momentum=0.9):
        self.gamma = np.ones((num_features, 1))   # Learnable scale
        self.beta = np.zeros((num_features, 1))    # Learnable shift
        self.epsilon = epsilon
        self.momentum = momentum
        # Running statistics for inference
        self.running_mean = np.zeros((num_features, 1))
        self.running_var = np.ones((num_features, 1))

    def forward(self, Z, training=True):
        """Normalize Z across the batch dimension (axis=1)."""
        if training:
            # Compute batch statistics
            self.mu = np.mean(Z, axis=1, keepdims=True)        # (n, 1)
            self.var = np.var(Z, axis=1, keepdims=True)         # (n, 1)
            # Normalize
            self.Z_norm = (Z - self.mu) / np.sqrt(self.var + self.epsilon)
            # Scale and shift
            out = self.gamma * self.Z_norm + self.beta
            # Update running stats (for inference)
            self.running_mean = self.momentum * self.running_mean + (1 - self.momentum) * self.mu
            self.running_var = self.momentum * self.running_var + (1 - self.momentum) * self.var
        else:
            # Use running statistics at inference
            Z_norm = (Z - self.running_mean) / np.sqrt(self.running_var + self.epsilon)
            out = self.gamma * Z_norm + self.beta
        return out



In [ ]:
# ── Demonstration ──
print("--- Batch Normalization Effect ---")
np.random.seed(42)

# Simulate pre-activations with large mean and variance
Z = np.random.randn(4, 8) * 10 + 50  # 4 features, 8 samples
print(f"Before BN → Mean: {np.mean(Z, axis=1).round(2)}, Var: {np.var(Z, axis=1).round(2)}")

bn = BatchNorm1D(num_features=4)
Z_bn = bn.forward(Z, training=True)
print(f"After  BN → Mean: {np.mean(Z_bn, axis=1).round(4)}, Var: {np.var(Z_bn, axis=1).round(4)}")
print("[Result] Mean ≈ 0, Variance ≈ 1 after normalization.")



In [ ]:
# ── Inference mode uses running statistics ──
print("\n--- Inference Mode (Single Sample) ---")
Z_test = np.random.randn(4, 1) * 10 + 50
Z_test_bn = bn.forward(Z_test, training=False)
print(f"Test input:  {Z_test.flatten().round(2)}")
print(f"Normalized:  {Z_test_bn.flatten().round(4)}")
